In [0]:
from pyspark.sql import functions as F
import pandas as pd
import io
import os
import time
import zipfile
import shutil
from pathlib import Path
from tqdm import tqdm

In [0]:
%sql
USE CATALOG prd_mega;
USE SCHEMA scolom15;
SELECT current_catalog() AS catalog, current_schema() AS schema;

In [0]:
BRONZE_TABLE  = "prd_mega.scolom15.bronze_validaciones_from2016to2019"
# Inspect Columns of BRONZE_TABLE
spark.table(BRONZE_TABLE).columns

In [0]:
# Show random observations for fecha_transaccion
# List date variables 
date_vars = ["fecha_transaccion", "clearing_date"]



In [0]:
# List date variables 
date_vars = ["fecha_transaccion", "clearing_date"]

# Build df_with_type_fecha ONCE with both type columns.
# First, trim whitespace from date columns (handles cases like "14-06-2018 ")
df_with_type_fecha = spark.table(BRONZE_TABLE)
for var in date_vars:
    df_with_type_fecha = df_with_type_fecha.withColumn(var, F.trim(F.col(var)))

# Classify the format of each date variable
for var in date_vars:
    df_with_type_fecha = df_with_type_fecha.withColumn(
        f"type_{var}",
        F.when(F.col(var).rlike(r"^(2016|2017|2018|2019)\d{10}$"), F.lit("YYYYMMDDHHmmss"))
        .when(F.col(var).rlike(r"^\d{4}/\d{2}/\d{2} \d{2}:\d{2}:\d{2}$"), F.lit("YYYY/MM/DD HH:mm:ss"))
        .when(F.col(var).rlike(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"), F.lit("YYYY-MM-DD HH:mm:ss"))
        .when(F.col(var).rlike(r"^\d{4}/\d{2}/\d{2} \d{2}:\d{2}:\d{2} (UTC)$"), F.lit("YYYY/MM/DD HH:mm:ss UTC"))
        .when(F.col(var).rlike(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2} (UTC)$"), F.lit("YYYY-MM-DD HH:mm:ss UTC"))
        .when(F.col(var).rlike(r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}:\d{2}$"), F.lit("DD/MM/YYYY HH:mm:ss"))
        .when(F.col(var).rlike(r"^\d{2}-\d{2}-\d{4} \d{2}:\d{2}:\d{2}$"), F.lit("DD-MM-YYYY HH:mm:ss"))
        .when(F.col(var).rlike(r"^(2016|2017|2018|2019)\d{4}$"), F.lit("YYYYMMDD"))
        .when(F.col(var).rlike(r"^\d{4}/\d{2}/\d{2}$"), F.lit("YYYY/MM/DD"))
        .when(F.col(var).rlike(r"^\d{2}/\d{2}/\d{4}$"), F.lit("DD/MM/YYYY"))
        .when(F.col(var).rlike(r"^\d{4}-\d{2}-\d{2}$"), F.lit("YYYY-MM-DD"))
        .when(F.col(var).rlike(r"^\d{2}-\d{2}-\d{4}$"), F.lit("DD-MM-YYYY"))
        .otherwise(F.lit("unknown"))
    )

# Display exploration for each variable
for var in date_vars:
    print(f"\n{'='*60}\nVariable: {var}\n{'='*60}")
    display(
        df_with_type_fecha
        .select(var, f"type_{var}")
        .orderBy(F.col("balance_after").desc())
        .limit(10)
    )

    # Count unknowns
    unknown_ft = df_with_type_fecha.filter(F.col(f"type_{var}") == "unknown").count()
    print(f"Number of unknown type_{var}: {unknown_ft}")

    if unknown_ft > 0:
        display(
            df_with_type_fecha
            .select(var)
            .where(F.col(f"type_{var}") == "unknown")
            .orderBy(F.col(var).desc())
            .limit(10)
        )


In [0]:
# Time-bearing formats: populate both _timestamp and _date
# Date-only formats:  leave _timestamp null, populate only _date
_time_formats = {
    "YYYYMMDDHHmmss":          "yyyyMMddHHmmss",
    "YYYY/MM/DD HH:mm:ss":     "yyyy/MM/dd HH:mm:ss",
    "YYYY-MM-DD HH:mm:ss":     "yyyy-MM-dd HH:mm:ss",
    "YYYY/MM/DD HH:mm:ss UTC": "yyyy/MM/dd HH:mm:ss z",
    "YYYY-MM-DD HH:mm:ss UTC": "yyyy-MM-dd HH:mm:ss z",
    "DD/MM/YYYY HH:mm:ss":     "dd/MM/yyyy HH:mm:ss",
    "DD-MM-YYYY HH:mm:ss":     "dd-MM-yyyy HH:mm:ss",
}
_date_only_formats = {
    "YYYYMMDD":    "yyyyMMdd",
    "YYYY/MM/DD":  "yyyy/MM/dd",
    "YYYY-MM-DD":  "yyyy-MM-dd",
    "DD-MM-YYYY":  "dd-MM-yyyy",
    "DD/MM/YYYY":  "dd/MM/yyyy",
}

# Build all timestamp/date columns cumulatively on the unified df_with_type_fecha
# (which already has trimmed values + both type_fecha_transaccion and type_clearing_date)
df_with_transaction_date = df_with_type_fecha

for var in date_vars:
    # _timestamp: only for time-bearing formats
    ts_expr = F.lit(None).cast("timestamp")
    for label, fmt in _time_formats.items():
        ts_expr = F.when(F.col(f"type_{var}") == label,
                         F.to_timestamp(F.col(var), fmt)).otherwise(ts_expr)

    # _date: always populated when format is known
    dt_expr = F.lit(None).cast("date")
    for label, fmt in _time_formats.items():
        dt_expr = F.when(F.col(f"type_{var}") == label,
                         F.to_date(F.col(var), fmt)).otherwise(dt_expr)
    for label, fmt in _date_only_formats.items():
        dt_expr = F.when(F.col(f"type_{var}") == label,
                         F.to_date(F.col(var), fmt)).otherwise(dt_expr)

    df_with_transaction_date = (
        df_with_transaction_date
        .withColumn(f"{var}_timestamp", ts_expr)
        .withColumn(f"{var}_date", dt_expr)
    )

# Preview and sanity check for each variable
for var in date_vars:
    types = df_with_transaction_date.select(f"type_{var}").distinct().collect()
    print(f"\n{'='*60}\n{var} — formats found: {[r[0] for r in types]}\n{'='*60}")
#    Uncomment for checking that all different formats look ok
#    for t in types:
#        display(
#            df_with_transaction_date
#            .select(var, f"type_{var}", f"{var}_timestamp", f"{var}_date")
#            .filter(F.col(f"type_{var}") == t[0])
#            .orderBy(F.col(f"{var}_date").desc())
#            .limit(10)
#        )

    null_ts = df_with_transaction_date.filter(F.col(f"{var}_timestamp").isNull()).count()
    null_dt = df_with_transaction_date.filter(F.col(f"{var}_date").isNull()).count()
    total   = df_with_transaction_date.count()
    print(f"Total rows          : {total:,}")
    print(f"Null {var}_timestamp : {null_ts:,}  ({100*null_ts/total:.1f}%)  ← expected for date-only rows")
    print(f"Null {var}_date      : {null_dt:,}  ({100*null_dt/total:.1f}%)  ← should be 0 or only 'unknown' rows")

In [0]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import numpy as np
import calendar

# ── 1. Aggregate to daily counts ─────────────────────────────────────────────
daily_pd = (
    df_with_transaction_date
    .filter(F.col("clearing_date_date").isNotNull())
    .groupBy("clearing_date_date")
    .count()
    .withColumn("year",  F.year("clearing_date_date"))
    .withColumn("month", F.month("clearing_date_date"))
    .withColumn("day",   F.dayofmonth("clearing_date_date"))
    .toPandas()
)

years = sorted(daily_pd["year"].unique())
MONTH_LABELS = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]

fig, axes = plt.subplots(len(years), 1, figsize=(22, 4 * len(years)))
if len(years) == 1:
    axes = [axes]

for ax, year in zip(axes, years):
    yd = daily_pd[daily_pd["year"] == year]

    # ── 2. Build 12 × 31 count grid ──────────────────────────────────────────
    # NaN  → invalid calendar day (e.g. Feb-30)  → white
    # 0    → valid day with no data              → red
    # n>0  → n transactions                      → gray gradient
    grid = np.full((12, 31), np.nan)
    for m in range(1, 13):
        n_days = calendar.monthrange(year, m)[1]
        grid[m - 1, :n_days] = 0           # valid days start as "no data"
    for _, row in yd.iterrows():
        grid[int(row["month"]) - 1, int(row["day"]) - 1] = row["count"]

    max_count = yd["count"].max() if not yd.empty else 1

    # ── 3. Gray layer: valid days with data ───────────────────────────────────
    gray_data = np.ma.masked_where((np.isnan(grid)) | (grid == 0), grid)
    gray_cmap = plt.cm.Greys
    ax.imshow(gray_data, aspect="auto", cmap=gray_cmap,
              vmin=0, vmax=max_count, interpolation="nearest")

    # ── 4. Red overlay: valid days with no data ───────────────────────────────
    red_rgba = np.zeros((12, 31, 4))
    red_rgba[(~np.isnan(grid)) & (grid == 0)] = [1, 0, 0, 1]
    ax.imshow(red_rgba, aspect="auto", interpolation="nearest")

    # ── 5. Colorbar ───────────────────────────────────────────────────────────
    sm = plt.cm.ScalarMappable(
        cmap="Greys",
        norm=mcolors.Normalize(vmin=0, vmax=max_count)
    )
    cbar = plt.colorbar(sm, ax=ax, label="Transactions", fraction=0.015, pad=0.01)
    cbar.ax.yaxis.set_label_position("left")

    # ── 6. Axes, grid lines, labels ───────────────────────────────────────────
    ax.set_yticks(range(12))
    ax.set_yticklabels(MONTH_LABELS, fontsize=9)
    ax.set_xticks(range(31))
    ax.set_xticklabels(range(1, 32), fontsize=8)
    ax.set_xlabel("Day of month", fontsize=10)
    ax.set_title(str(year), fontsize=13, fontweight="bold", pad=8)

    ax.set_xticks(np.arange(-0.5, 31, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 12, 1), minor=True)
    ax.grid(which="minor", color="lightgray", linewidth=0.4)
    ax.tick_params(which="minor", bottom=False, left=False)

    red_p   = mpatches.Patch(color="red",  label="No data (valid day)")
    white_p = mpatches.Patch(facecolor="white", edgecolor="lightgray", label="Invalid date")
    ax.legend(handles=[red_p, white_p], loc="lower right", fontsize=8, framealpha=0.85)

plt.suptitle("Daily Transaction Heatmap  (month × day-of-month)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [0]:
# ── Clean emisor → issuer_id ──────────────────────────────────────────────────
# Strategy: build on df_with_transaction_date, normalize known encoding variants
# using .contains() on the issuer code prefix, and flag corrupted rows.

# Define canonical mapping: (code_prefix) → canonical_name
_issuer_map = {
    "(1900001)": "(1900001) Angelcom Card",
    "(3200401)": "(3200401) Colpatria",
    "(3101000)": "(3101000) Bogota Card (Citizen)",
    "(3200201)": "(3200201) AV Villas",
    "(3200101)": "(3200101) Bancolombia",
    "(3200501)": "(3200501) Codensa (Master Card)",
    "(3200301)": "(3200301) Davivienda",
    "(3200502)": "(3200502) Codensa (Sin Franquicia)",
    "(3200701)": "(3200701) RappiPay-Daviplata",
    "(3200601)": "(3200601) Itau",  # normalizes Itaú / ItaÃº / Ita��
}

# Build the when-chain: if emisor contains the code prefix → canonical name
issuer_expr = F.lit(None).cast("string")
for prefix, canonical in _issuer_map.items():
    issuer_expr = F.when(F.col("emisor").contains(prefix), F.lit(canonical)).otherwise(issuer_expr)

df_clean = df_with_transaction_date.withColumn("issuer_id", issuer_expr)

# Preview: compare raw vs cleaned
print("── issuer_id mapping preview ──")
display(
    df_clean
    .select("emisor", "issuer_id")
    .distinct()
    .orderBy("issuer_id")
)

# Check unmapped rows (nulls in issuer_id)
unmapped = df_clean.filter(F.col("issuer_id").isNull()).count()
total = df_clean.count()
print(f"\nUnmapped rows (issuer_id is null): {unmapped:,} / {total:,} ({100*unmapped/total:.2f}%)")

In [0]:
# ── Clean operator → operator_id ──────────────────────────────────────────────────
# Strategy: build on df_with_transaction_date, normalize known encoding variants
# using .contains() on the operator code prefix, and flag corrupted rows.

# Define canonical mapping: (code_prefix) → canonical_name
_operator_map = {
    "(001)": "(001) CONSORCIO EXPRESS USAQUEN",
    "(002)": "(002) MASIVO CAPITAL SUBA ORIENTAL",
    "(004)": "(004) ESTE ES MI BUS CALLE 80",
    "(005)": "(005) GMOVIL",
    "(007)": "(007) ETIB",
    "(008)": "(008) SUMA",
    "(009)": "(009) TRANZIT",
    "(011)": "(011) CONSORCIO EXPRESS SAN CRISTOBAL",
    "(012)": "(012) MASIVO CAPITAL KENNEDY",
    "(014)": "(014) ESTE ES MI BUS TINTAL ZONA FRANCA",
    "(201)": "(201) Trunk agency",
    "(2054)": "(2054) HABITUAL_2015-11-09",
}

# Build the when-chain: if operator contains the code prefix → canonical name
operator_expr = F.lit(None).cast("string")
for prefix, canonical in _operator_map.items():
    operator_expr = F.when(F.col("operator").contains(prefix), F.lit(canonical)).otherwise(operator_expr)

df_clean = df_with_transaction_date.withColumn("operator_id", operator_expr)

# Preview: compare raw vs cleaned
print("── operator_id mapping preview ──")
display(
    df_clean
    .select("operator", "operator_id")
    .distinct()
    .orderBy("operator_id")
)

# Check unmapped rows (nulls in operator_id)
unmapped = df_clean.filter(F.col("operator_id").isNull()).count()
total = df_clean.count()
print(f"\nUnmapped rows (operator_id is null): {unmapped:,} / {total:,} ({100*unmapped/total:.2f}%)")

In [0]:
# ── Clean line → line_id ──────────────────────────────────────────────────
# Strategy: trim whitespace and remove trailing slashes

df_clean = df_with_transaction_date.withColumn(
    "line_id",
    F.regexp_replace(F.trim(F.col("line")), r"/$", "")
)

# Preview: compare raw vs cleaned
print("── line_id mapping preview ──")
display(
    df_clean
    .select("line", "line_id")
    .distinct()
    .orderBy("line_id")
)

# Check null rows
unmapped = df_clean.filter(F.col("line_id").isNull()).count()
total = df_clean.count()
print(f"\nNull line_id rows: {unmapped:,} / {total:,} ({100*unmapped/total:.2f}%)")

In [0]:
# ── Clean station, station_access, machine, vehicle_id, route, fare_type ──
# Strategy: trim whitespace and remove trailing slashes for each variable

vars_to_clean = ['station', 'station_access', 'machine', 'vehicle_id', 'route', 'fare_type']

df_clean = df_with_transaction_date
for var in vars_to_clean:
    df_clean = df_clean.withColumn(
        f"{var}_id",
        F.regexp_replace(F.trim(F.col(var)), r"/$", "")
    )

# Preview distinct counts before/after for each variable
print("── Distinct value counts: raw vs cleaned ──")
for var in vars_to_clean:
    raw_count = df_clean.select(var).distinct().count()
    clean_count = df_clean.select(f"{var}_id").distinct().count()
    print(f"  {var}: {raw_count} distinct → {var}_id: {clean_count} distinct")

In [0]:
# ── Clean phase → phase_id ──────────────────────────────────────────────────
# "Phase 3" if phase contains "3", null otherwise

df_clean = df_with_transaction_date.withColumn(
    "phase_id",
    F.when(F.col("phase").contains("3"), F.lit("Phase 3")).otherwise(F.lit(None))
)

# Preview
print("── phase_id mapping preview ──")
display(
    df_clean
    .select("phase", "phase_id")
    .distinct()
    .orderBy("phase_id")
)

unmapped = df_clean.filter(F.col("phase_id").isNull()).count()
total = df_clean.count()
print(f"\nNull phase_id rows: {unmapped:,} / {total:,} ({100*unmapped/total:.2f}%)")

In [0]:
# ── Clean peak_hour → peak_hour_id ──────────────────────────────────────────
# "Peak Time" if == "Peak Time", "Non Peak Time" if == "Non Peak Time", null otherwise

df_clean = df_with_transaction_date.withColumn(
    "peak_hour_id",
    F.when(F.col("peak_hour") == "Peak Time", F.lit("Peak Time"))
     .when(F.col("peak_hour") == "Non Peak Time", F.lit("Non Peak Time"))
     .otherwise(F.lit(None))
)

# Preview
print("── peak_hour_id mapping preview ──")
display(
    df_clean
    .select("peak_hour", "peak_hour_id")
    .distinct()
    .orderBy("peak_hour_id")
)

unmapped = df_clean.filter(F.col("peak_hour_id").isNull()).count()
total = df_clean.count()
print(f"\nNull peak_hour_id rows: {unmapped:,} / {total:,} ({100*unmapped/total:.2f}%)")

In [0]:
# ── Clean card_type → card_type_id ──────────────────────────────────────────────────

# Define canonical mapping:
_card_type_map = {
    "Plus": "TuLlave Plus",
    "sica": "TuLlave Basica",
    "Angel": "Angelcom",
}

# Build the when-chain: if card_type contains the keyword → canonical name
card_type_expr = F.lit(None).cast("string")
for prefix, canonical in _card_type_map.items():
    card_type_expr = F.when(F.col("card_type").contains(prefix), F.lit(canonical)).otherwise(card_type_expr)

df_clean = df_with_transaction_date.withColumn("card_type_id", card_type_expr)

# Preview: compare raw vs cleaned
print("── card_type_id mapping preview ──")
display(
    df_clean
    .select("card_type", "card_type_id")
    .distinct()
    .orderBy("card_type_id")
)

# Check unmapped rows (nulls in card_type_id)
unmapped = df_clean.filter(F.col("card_type_id").isNull()).count()
total = df_clean.count()
print(f"\nUnmapped rows (card_type_id is null): {unmapped:,} / {total:,} ({100*unmapped/total:.2f}%)")

In [0]:
# ── Clean vehicle_type → vehicle_type_id ──────────────────────────────────────
# "(02) Urbano" if vehicle_type == "(02) Urbano", null otherwise

df_clean = df_with_transaction_date.withColumn(
    "vehicle_type_id",
    F.when(F.col("vehicle_type") == "(02) Urbano", F.lit("(02) Urbano")).otherwise(F.lit(None))
)

# Preview
print("── vehicle_type_id mapping preview ──")
display(
    df_clean
    .select("vehicle_type", "vehicle_type_id")
    .distinct()
    .orderBy("vehicle_type_id")
)

unmapped = df_clean.filter(F.col("vehicle_type_id").isNull()).count()
total = df_clean.count()
print(f"\nNull vehicle_type_id rows: {unmapped:,} / {total:,} ({100*unmapped/total:.2f}%)")

In [0]:
# ── Clean account_name → account_name_id ──────────────────────────────────────
# Strategy: map by code prefix to canonical name.
# NOTE: some codes have CONFLICTING names across systems — flagged below.

_account_name_map = {
    "(000)": "(000) Unknown",
    "(001) Adulto": "(001) Adulto",           # ⚠️ also "(001) Anonymous" — mapped separately
    "(001) Anonymous": "(001) Anonymous",
    "(002)": "(002) Adulto Mayor",
    "(003) Estudiantil": "(003) Estudiantil",  # ⚠️ also "(003) Capital" — mapped separately
    "(003) Capital": "(003) Capital",
    "(004)": "(004) Menor de Edad",
    "(005)": "(005) Discapacidad",
    "(006) Apoyo": "(006) Apoyo Ciudadano",    # ⚠️ also "(006) Discapacitados" — mapped separately
    "(006) Discapacitado": "(006) Discapacitados",
    "(008)": "(008) Etnico",
    "(014)": "(014) Usuario frecuente",
    "(017)": "(017) Discapacitado Monedero",
    "(018)": "(018) Universitaria",
    "(021)": "(021) Tarjeta Ciudadana",
    "(022)": "(022) Empresarial TM",
    "(023)": "(023) Empresarial Davivienda",
    "(024)": "(024) Empresarial Colsubsidio",
    "(025)": "(025) Empresarial Compensar",
    "(026)": "(026) Empresarial AV Villas",
    "(027)": "(027) Club Universitario",
    "(029)": "(029) Empresarial Banco de Bogota",
    "(030)": "(030) Capital monedero",
    "(032)": "(032) Empresarial Daviplata",
    "(033)": "(033) Empresarial People Pass",
    "(035)": "(035) Empresarial AV Villas Credito",
    "(036)": "(036) Empresarial Colpatria",
    "(041)": "(041) Empresarial Cercanos",
    "(044)": "(044) Empresarial CIS",
    "(101)": "(101) Adulto PV",
}

# Build the when-chain: use .contains() on the MOST SPECIFIC keys first
# (to avoid "(001) Adulto" being caught by a generic "(001)" prefix)
# Sort keys longest-first so specific matches take priority
account_expr = F.lit(None).cast("string")
for key, canonical in sorted(_account_name_map.items(), key=lambda x: len(x[0])):
    account_expr = F.when(F.col("account_name").contains(key), F.lit(canonical)).otherwise(account_expr)

df_clean = df_with_transaction_date.withColumn("account_name_id", account_expr)

# Preview: compare raw vs cleaned
print("── account_name_id mapping preview ──")
display(
    df_clean
    .select("account_name", "account_name_id")
    .distinct()
    .orderBy("account_name_id")
)

# Check unmapped rows
unmapped = df_clean.filter(F.col("account_name_id").isNull()).count()
total = df_clean.count()
print(f"\nUnmapped rows (account_name_id is null): {unmapped:,} / {total:,} ({100*unmapped/total:.2f}%)")


 'cardnumber',
 'balance_before',
 'value',
 'balance_after',
 'system',
 'day_group_type',
 '_source_file',
 '_header_group',
 '_transform_format',
 '_ingestion_ts']

In [0]:
numvars = ['cardnumber', 'balance_before', 'value', 'balance_after']

# For each numvar, check that it contains no letters, then destring to double
for numvar in numvars:
    print(f"── {numvar} value check ──")
    display(
        df_with_transaction_date
        .select(numvar)
        .filter(~F.col(numvar).rlike("[A-Za-z]") | F.col(numvar).isNull())
        .withColumn(f"{numvar}_double", F.col(numvar).cast("double"))
        .groupBy(numvar, f"{numvar}_double")
        .count()
        .orderBy(numvar)
    )

In [0]:
# Count how many time each value appears
spark.table(BRONZE_TABLE).select('card_type').groupBy('card_type').count().display()


In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# CONSOLIDATION: chain ALL cleaning transformations into a single df_silver
# ══════════════════════════════════════════════════════════════════════════════

# ── 1. Issuer ─────────────────────────────────────────────────────────────────
_issuer_map = {
    "(1900001)": "(1900001) Angelcom Card",
    "(3200401)": "(3200401) Colpatria",
    "(3101000)": "(3101000) Bogota Card (Citizen)",
    "(3200201)": "(3200201) AV Villas",
    "(3200101)": "(3200101) Bancolombia",
    "(3200501)": "(3200501) Codensa (Master Card)",
    "(3200301)": "(3200301) Davivienda",
    "(3200502)": "(3200502) Codensa (Sin Franquicia)",
    "(3200701)": "(3200701) RappiPay-Daviplata",
    "(3200601)": "(3200601) Itau",
}
issuer_expr = F.lit(None).cast("string")
for prefix, canonical in _issuer_map.items():
    issuer_expr = F.when(F.col("emisor").contains(prefix), F.lit(canonical)).otherwise(issuer_expr)

# ── 2. Operator ───────────────────────────────────────────────────────────────
_operator_map = {
    "(001)": "(001) CONSORCIO EXPRESS USAQUEN",
    "(002)": "(002) MASIVO CAPITAL SUBA ORIENTAL",
    "(004)": "(004) ESTE ES MI BUS CALLE 80",
    "(005)": "(005) GMOVIL",
    "(007)": "(007) ETIB",
    "(008)": "(008) SUMA",
    "(009)": "(009) TRANZIT",
    "(011)": "(011) CONSORCIO EXPRESS SAN CRISTOBAL",
    "(012)": "(012) MASIVO CAPITAL KENNEDY",
    "(014)": "(014) ESTE ES MI BUS TINTAL ZONA FRANCA",
    "(201)": "(201) Trunk agency",
    "(2054)": "(2054) HABITUAL_2015-11-09",
}
operator_expr = F.lit(None).cast("string")
for prefix, canonical in _operator_map.items():
    operator_expr = F.when(F.col("operator").contains(prefix), F.lit(canonical)).otherwise(operator_expr)

# ── 3. Card type ──────────────────────────────────────────────────────────────
_card_type_map = {
    "Plus": "TuLlave Plus",
    "sica": "TuLlave Basica",
    "Angel": "Angelcom",
}
card_type_expr = F.lit(None).cast("string")
for prefix, canonical in _card_type_map.items():
    card_type_expr = F.when(F.col("card_type").contains(prefix), F.lit(canonical)).otherwise(card_type_expr)

# ── 4. Build df_silver with all transformations ───────────────────────────────
df_silver = (
    df_with_transaction_date
    # Categorical mappings
    .withColumn("issuer_id", issuer_expr)
    .withColumn("operator_id", operator_expr)
    .withColumn("card_type_id", card_type_expr)
    # Trim + remove trailing slashes
    .withColumn("line_id", F.regexp_replace(F.trim(F.col("line")), r"/$", ""))
    .withColumn("station_id", F.regexp_replace(F.trim(F.col("station")), r"/$", ""))
    .withColumn("station_access_id", F.regexp_replace(F.trim(F.col("station_access")), r"/$", ""))
    .withColumn("machine_id", F.regexp_replace(F.trim(F.col("machine")), r"/$", ""))
    .withColumn("vehicle_id_clean", F.regexp_replace(F.trim(F.col("vehicle_id")), r"/$", ""))
    .withColumn("route_id", F.regexp_replace(F.trim(F.col("route")), r"/$", ""))
    .withColumn("fare_type_id", F.regexp_replace(F.trim(F.col("fare_type")), r"/$", ""))
    # Conditional mappings
    .withColumn("phase_id", F.when(F.col("phase").contains("3"), F.lit("Phase 3")).otherwise(F.lit(None)))
    .withColumn("peak_hour_id", 
        F.when(F.col("peak_hour") == "Peak Time", F.lit("Peak Time"))
         .when(F.col("peak_hour") == "Non Peak Time", F.lit("Non Peak Time"))
         .otherwise(F.lit(None)))
    .withColumn("vehicle_type_id", 
        F.when(F.col("vehicle_type") == "(02) Urbano", F.lit("(02) Urbano")).otherwise(F.lit(None)))
    # Select only clean columns for silver
    .select(
        # Date columns (already cleaned)
        "fecha_transaccion_timestamp",
        "fecha_transaccion_date",
        "clearing_date_timestamp",
        "clearing_date_date",
        # Cleaned categorical columns
        "issuer_id",
        "operator_id",
        "line_id",
        "station_id",
        "station_access_id",
        "machine_id",
        "phase_id",
        "peak_hour_id",
        "vehicle_id_clean",
        "route_id",
        "fare_type_id",
        "card_type_id",
        "vehicle_type_id",
        # Numeric / pass-through columns (keep as-is for now)
        "account_name",
        "cardnumber",
        "balance_before",
        "value",
        "balance_after",
        "system",
        "day_group_type",
        # Metadata
        "_source_file",
        "_header_group",
        "_transform_format",
        "_ingestion_ts",
    )
)

# ── 5. Summary ────────────────────────────────────────────────────────────────
print(f"df_silver columns: {len(df_silver.columns)}")
print(f"Schema:")
df_silver.printSchema()
print(f"\nRow count: {df_silver.count():,}")